# Advanced Certification Programme in AI and MLOps
## A Program by IISc and TalentSprint
### Notebook 2: Leveraging LLMs for Querying Insights (Data Loading)

## Learning Objectives

At the end of the experiment, you will be able to:

* load data from pickle files
* store the data in a SQLite database
* use LLMs to create chains for querying insights from this database

## Data Description

There are 2 pickle files which will be downloaded once you run the below cell:

> - *constituent_stock_prices.pkl*
>
>    It contains information related to stock prices - open, high, low, close, volume

> - *constituent_stock_fundamentals.pkl*
>
>    It contains information related to fundamentals - income statement, balancesheet statement, cashflow statement


In [ ]:
#@title Download the pickle files
from IPython.display import clear_output
!gdown https://drive.google.com/uc?id=1c3eVGtfBlg9slenmz_DJkJJlH0zwKzlZ
!gdown https://drive.google.com/uc?id=1U8p04e43oHFh_tx-nNy5AGxmQtceBEqn
clear_output()

!ls | grep '.pkl'

# Load Constituent Stock Prices data

In [ ]:
import pickle
import pandas as pd

csp = pickle.load(open('/content/constituent_stock_prices.pkl', 'rb'))

In [ ]:
type(csp)

In [ ]:
csp.keys()    # each key contains a dataframe

In [ ]:
len(csp.keys())

In [ ]:
rows = []
cols = []

for key in csp.keys():
    print(csp[key].shape)
    rows.append(csp[key].shape[0])
    cols.append(csp[key].shape[1])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 4))
plt.bar(csp.keys(), rows)
plt.xticks(rotation=80)
plt.show()

In [ ]:
adani = csp['ADANIENT.NS']
print(adani.shape)
adani.head()

In [ ]:
tcs = csp['TCS.NS']
print(tcs.shape)
tcs.head()

In [ ]:
from matplotlib import pyplot as plt
tcs['Close'].plot(kind='line', figsize=(8, 4), title='Close')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
wipro = csp['WIPRO.NS']
print(wipro.shape)
wipro.head()

## **Combine Prices(OHLC) data**

In [ ]:
comb_df = pd.DataFrame(columns=['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Symbol'])

for key in csp.keys():
    tmp_df = csp[key].copy()
    tmp_df.reset_index(inplace=True)
    tmp_df['Symbol'] = key.split('.')[0]
    comb_df = pd.concat([comb_df, tmp_df], ignore_index=True)


In [ ]:
comb_df.head()

In [ ]:
comb_df.shape

In [ ]:
sum(rows)

In [ ]:
comb_df.tail()

In [ ]:
comb_df.to_csv('all_stock_prices.csv', index=False)

## **Create a SQLite Database (in-memory)**

In [ ]:
import sqlite3

In [ ]:
# Read DB (It will create if it doesn't exists)

conn = sqlite3.connect('stock_db.sqlite')
print("Opened database successfully");

In [ ]:
comb_df.columns

### Create table **`stock_prices`**

In [ ]:
# Create table 'stock_prices' in DB

conn.execute('''
CREATE TABLE IF NOT EXISTS stock_prices(
                      date DATE,
                      open DOUBLE,
                      high DOUBLE,
                      low DOUBLE,
                      close DOUBLE,
                      volume INT,
                      symbol VARCHAR(20));''')

conn.commit()

print("Table created successfully");

In [ ]:
# Show tables

cursor = conn.execute('''
SELECT name FROM sqlite_master WHERE type='table';
''')

for row in cursor:
    print(row)

### Insert data into **`stock_prices`** table

In [ ]:
def convert_date(date):
    yyyy = date.year
    mm = date.month
    dd = date.day
    if mm<10:
        mm = '0' + str(mm)

    if dd<10:
        dd = '0' + str(dd)
    return f"{yyyy}-{mm}-{dd}"

In [ ]:
convert_date(comb_df['Date'][0])

In [ ]:
comb_df['Date'] = comb_df['Date'].apply(convert_date)
comb_df.head(3)

In [ ]:
# Insert stock prices data

conn.executemany('''
INSERT INTO stock_prices (date, open, high, low, close, volume, symbol) VALUES (?, ?, ?, ?, ?, ?, ?)
''', comb_df.values)

conn.commit()

print("Data inserted successfully!")

### **Query the Database**

In [ ]:
# Show table content

cursor = conn.execute('''
SELECT * from stock_prices limit 10;
''')

for row in cursor:
    print(row)

In [ ]:
# Show table content

cursor = conn.execute('''
SELECT * from stock_prices WHERE symbol='WIPRO' limit 10;
''')

for row in cursor:
    print(row)

In [ ]:
# Show number of rows in table

cursor = conn.execute('''
SELECT count(*) from stock_prices;
''')

for row in cursor:
    print(row)

In [ ]:
# Show column info of table

cursor = conn.execute('''
PRAGMA table_info(stock_prices);
''')

for row in cursor:
    print(row)

## Load Constituent Stock Fundamentals data

In [ ]:
import pickle

csf = pickle.load(open('/content/constituent_stock_fundamentals.pkl', 'rb'))

In [ ]:
type(csf)

In [ ]:
csf.keys()     # each key contains a dict with keys ['income_statement', 'balancesheet_statement', 'cashflow_statement']


In [ ]:
len(csf.keys())

In [ ]:
d1 = csf['ADANIENT.NS']
type(d1)

In [ ]:
d1.keys()

In [ ]:
for key in d1.keys():
  print(key, type(d1[key]))

In [ ]:
type(d1['income_statement'])

In [ ]:
for key in csf.keys():
    print(f"{key:<18}", end=' ')
    print(pd.DataFrame(csf[key]['income_statement']).shape, end=' ')
    print(pd.DataFrame(csf[key]['balancesheet_statement']).shape, end=' ')
    print(pd.DataFrame(csf[key]['cashflow_statement']).shape)


In [ ]:
dd1 = pd.DataFrame(d1['income_statement'])
print(dd1.shape)
dd1.head()

In [ ]:
dd1.dtypes

In [ ]:
dd1.columns

In [ ]:
dd1['calendarYear'].nunique(), dd1['calendarYear'].min(), dd1['calendarYear'].max()

In [ ]:
dd2 = pd.DataFrame(d1['balancesheet_statement'])
print(dd2.shape)
dd2.head()

In [ ]:
dd2.dtypes

In [ ]:
dd2.columns

In [ ]:
dd3 = pd.DataFrame(d1['cashflow_statement'])
print(dd3.shape)
dd3.head()

In [ ]:
dd3.dtypes

In [ ]:
dd3.columns

## Combine 'income_statement' data

In [ ]:
colm_list = pd.DataFrame(csf['ADANIENT.NS']['income_statement']).columns.tolist()
print(colm_list)

In [ ]:
colm_list = pd.DataFrame(csf['ADANIENT.NS']['income_statement']).columns.tolist()

income_df = pd.DataFrame(columns=colm_list)

for key in csf.keys():
    tmp_df = pd.DataFrame(csf[key]['income_statement']).copy()
    income_df = pd.concat([income_df, tmp_df], ignore_index=True)

income_df['symbol'] = income_df['symbol'].apply(lambda x: x.split('.')[0])


In [ ]:
income_df.head()

In [ ]:
income_df.shape

In [ ]:
cnt = 0
for key in csf.keys():
    tmp_df = pd.DataFrame(csf[key]['income_statement']).copy()
    cnt += tmp_df.shape[0]

print(cnt)

In [ ]:
dd1.dtypes.values

In [ ]:
income_df.dtypes.values

In [ ]:
for i in range(len(income_df.columns)):
    if income_df.dtypes.values[i] !=  dd1.dtypes.values[i]:
        income_df[income_df.columns[i]] = income_df[income_df.columns[i]].astype(dd1.dtypes.values[i])

    #print(f"{income_df.columns[i]:<20}  {income_df.dtypes.values[i]}  {dd1.dtypes.values[i]}")


In [ ]:
for i in range(len(income_df.columns)):
    print(f"{income_df.columns[i]:<20}  {income_df.dtypes.values[i]}  {dd1.dtypes.values[i]}")


In [ ]:
income_df.to_csv('income_statement.csv', index=False)

## Combine 'balancesheet_statement' data

In [ ]:
colm_list = pd.DataFrame(csf['ADANIENT.NS']['balancesheet_statement']).columns.tolist()
print(colm_list)

In [ ]:
colm_list = pd.DataFrame(csf['ADANIENT.NS']['balancesheet_statement']).columns.tolist()

balancesheet_df = pd.DataFrame(columns=colm_list)

for key in csf.keys():
    tmp_df = pd.DataFrame(csf[key]['balancesheet_statement']).copy()
    balancesheet_df = pd.concat([balancesheet_df, tmp_df], ignore_index=True)

balancesheet_df['symbol'] = balancesheet_df['symbol'].apply(lambda x: x.split('.')[0])


In [ ]:
balancesheet_df.head()

In [ ]:
balancesheet_df.shape

In [ ]:
cnt = 0
for key in csf.keys():
    tmp_df = pd.DataFrame(csf[key]['balancesheet_statement']).copy()
    cnt += tmp_df.shape[0]

print(cnt)

In [ ]:
dd2.dtypes.values

In [ ]:
balancesheet_df.dtypes.values

In [ ]:
for i in range(len(income_df.columns)):
    if balancesheet_df.dtypes.values[i] !=  dd2.dtypes.values[i]:
        balancesheet_df[balancesheet_df.columns[i]] = balancesheet_df[balancesheet_df.columns[i]].astype(dd2.dtypes.values[i])


In [ ]:
for i in range(len(income_df.columns)):
    print(f"{balancesheet_df.columns[i]:<35}  {balancesheet_df.dtypes.values[i]}  {dd2.dtypes.values[i]}")


In [ ]:
balancesheet_df.to_csv('balancesheet_statement.csv', index=False)

## Combine 'cashflow_statement' data

In [ ]:
colm_list = pd.DataFrame(csf['ADANIENT.NS']['cashflow_statement']).columns.tolist()
print(colm_list)

In [ ]:
colm_list = pd.DataFrame(csf['ADANIENT.NS']['cashflow_statement']).columns.tolist()

cashflow_df = pd.DataFrame(columns=colm_list)

for key in csf.keys():
    tmp_df = pd.DataFrame(csf[key]['cashflow_statement']).copy()
    cashflow_df = pd.concat([cashflow_df, tmp_df], ignore_index=True)

cashflow_df['symbol'] = cashflow_df['symbol'].apply(lambda x: x.split('.')[0])


In [ ]:
cashflow_df.head()

In [ ]:
cashflow_df.shape

In [ ]:
cnt = 0
for key in csf.keys():
    tmp_df = pd.DataFrame(csf[key]['cashflow_statement']).copy()
    cnt += tmp_df.shape[0]

print(cnt)

In [ ]:
dd3.dtypes.values

In [ ]:
cashflow_df.dtypes.values

In [ ]:
for i in range(len(income_df.columns)):
    if cashflow_df.dtypes.values[i] !=  dd3.dtypes.values[i]:
        cashflow_df[cashflow_df.columns[i]] = cashflow_df[cashflow_df.columns[i]].astype(dd3.dtypes.values[i])


In [ ]:
for i in range(len(income_df.columns)):
    print(f"{cashflow_df.columns[i]:<40}  {cashflow_df.dtypes.values[i]}  {dd3.dtypes.values[i]}")


In [ ]:
cashflow_df.to_csv('cashflow_statement.csv', index=False)

### Remove unnecessary columns

In [ ]:
income_df.drop(columns=['link', 'finalLink'], inplace=True)
balancesheet_df.drop(columns=['link', 'finalLink'], inplace=True)
cashflow_df.drop(columns=['link', 'finalLink'], inplace=True)

## Insert 'income_statement' data into SQLite DB

In [ ]:
income_table_template = '''
CREATE TABLE IF NOT EXISTS income_statement(\n'''

for i in range(len(income_df.columns)):
    if 'date' in str.lower(income_df.columns[i]):
        datatype = 'DATE'
    elif 'year' in str.lower(income_df.columns[i]):
        datatype = 'INTEGER'
    elif income_df.dtypes.values[i] == 'object':
        datatype = 'VARCHAR(20)'
    elif income_df.dtypes.values[i] == 'int64':
        datatype = 'INTEGER'
    elif income_df.dtypes.values[i] == 'float64':
        datatype = 'DOUBLE'
    else:
        datatype = 'VARCHAR(20)'

    income_table_template += f"      {income_df.columns[i]} {datatype}, \n"

income_table_template = income_table_template[:-3] + f"\n      );"


In [ ]:
print(income_table_template)

In [ ]:
# Create table 'income_statement' in DB

conn.execute(income_table_template)

conn.commit()

print("Table created successfully");

In [ ]:
# Show tables

cursor = conn.execute('''
SELECT name FROM sqlite_master WHERE type='table';
''')

for row in cursor:
    print(row)

In [ ]:
def convert_accepted_date(date):
    return date[:10]


In [ ]:
income_df['acceptedDate'][0]

In [ ]:
convert_accepted_date(income_df['acceptedDate'][0])

In [ ]:
income_df['acceptedDate'] = income_df['acceptedDate'].apply(convert_accepted_date)
income_df.head(3)

In [ ]:
income_insert_template = "\nINSERT INTO income_statement ("

for i in range(len(income_df.columns)):
    income_insert_template += f"{income_df.columns[i]}, "

income_insert_template = income_insert_template[:-2] + f") VALUES (?{', ?'*(len(income_df.columns)-1)})\n"

print(income_insert_template)

In [ ]:
# Insert stock prices data

conn.executemany(income_insert_template, income_df.values)

conn.commit()

print("Data inserted successfully!")

In [ ]:
# Show table content

cursor = conn.execute('''
SELECT * from income_statement limit 10;
''')

for row in cursor:
    print(row)

## Insert 'balancesheet_statement' data into SQLite DB

In [ ]:
balancesheet_table_template = '''
CREATE TABLE IF NOT EXISTS balancesheet_statement(\n'''

for i in range(len(balancesheet_df.columns)):
    if 'date' in str.lower(balancesheet_df.columns[i]):
        datatype = 'DATE'
    elif 'year' in str.lower(balancesheet_df.columns[i]):
        datatype = 'INTEGER'
    elif balancesheet_df.dtypes.values[i] == 'object':
        datatype = 'VARCHAR(20)'
    elif balancesheet_df.dtypes.values[i] == 'int64':
        datatype = 'INTEGER'
    elif balancesheet_df.dtypes.values[i] == 'float64':
        datatype = 'DOUBLE'
    else:
        datatype = 'VARCHAR(20)'

    balancesheet_table_template += f"      {balancesheet_df.columns[i]} {datatype}, \n"

balancesheet_table_template = balancesheet_table_template[:-3] + f"\n      );"


In [ ]:
print(balancesheet_table_template)

In [ ]:
# Create table 'income_statement' in DB

conn.execute(balancesheet_table_template)

conn.commit()

print("Table created successfully");

In [ ]:
# Show tables

cursor = conn.execute('''
SELECT name FROM sqlite_master WHERE type='table';
''')

for row in cursor:
    print(row)

In [ ]:
def convert_accepted_date(date):
    return date[:10]


In [ ]:
balancesheet_df['acceptedDate'][0]

In [ ]:
convert_accepted_date(balancesheet_df['acceptedDate'][0])

In [ ]:
balancesheet_df['acceptedDate'] = balancesheet_df['acceptedDate'].apply(convert_accepted_date)
balancesheet_df.head(3)

In [ ]:
balancesheet_insert_template = "\nINSERT INTO balancesheet_statement ("

for i in range(len(balancesheet_df.columns)):
    balancesheet_insert_template += f"{balancesheet_df.columns[i]}, "

balancesheet_insert_template = balancesheet_insert_template[:-2] + f") VALUES (?{', ?'*(len(balancesheet_df.columns)-1)})\n"

print(balancesheet_insert_template)

In [ ]:
# Insert stock prices data

conn.executemany(balancesheet_insert_template, balancesheet_df.values)

conn.commit()

print("Data inserted successfully!")

In [ ]:
# Show table content

cursor = conn.execute('''
SELECT * from balancesheet_statement limit 10;
''')

for row in cursor:
    print(row)

## Insert 'cashflow_statement' data into SQLite DB

In [ ]:
cashflow_table_template = '''
CREATE TABLE IF NOT EXISTS cashflow_statement(\n'''

for i in range(len(cashflow_df.columns)):
    if 'date' in str.lower(cashflow_df.columns[i]):
        datatype = 'DATE'
    elif 'year' in str.lower(cashflow_df.columns[i]):
        datatype = 'INTEGER'
    elif cashflow_df.dtypes.values[i] == 'object':
        datatype = 'VARCHAR(20)'
    elif cashflow_df.dtypes.values[i] == 'int64':
        datatype = 'INTEGER'
    elif cashflow_df.dtypes.values[i] == 'float64':
        datatype = 'DOUBLE'
    else:
        datatype = 'VARCHAR(20)'

    cashflow_table_template += f"      {cashflow_df.columns[i]} {datatype}, \n"

cashflow_table_template = cashflow_table_template[:-3] + f"\n      );"


In [ ]:
print(cashflow_table_template)

In [ ]:
# Create table 'income_statement' in DB

conn.execute(cashflow_table_template)

conn.commit()

print("Table created successfully");

In [ ]:
# Show tables

cursor = conn.execute('''
SELECT name FROM sqlite_master WHERE type='table';
''')

for row in cursor:
    print(row)

In [ ]:
def convert_accepted_date(date):
    return date[:10]


In [ ]:
cashflow_df['acceptedDate'][0]

In [ ]:
convert_accepted_date(cashflow_df['acceptedDate'][0])

In [ ]:
cashflow_df['acceptedDate'] = cashflow_df['acceptedDate'].apply(convert_accepted_date)
cashflow_df.head(3)

In [ ]:
cashflow_insert_template = "\nINSERT INTO cashflow_statement ("

for i in range(len(cashflow_df.columns)):
    cashflow_insert_template += f"{cashflow_df.columns[i]}, "

cashflow_insert_template = cashflow_insert_template[:-2] + f") VALUES (?{', ?'*(len(cashflow_df.columns)-1)})\n"

print(cashflow_insert_template)

In [ ]:
# Insert stock prices data

conn.executemany(cashflow_insert_template, cashflow_df.values)

conn.commit()

print("Data inserted successfully!")

In [ ]:
# Show table content

cursor = conn.execute('''
SELECT * from cashflow_statement limit 10;
''')

for row in cursor:
    print(row)

In [ ]:
## Once done, close the connection:

conn.close()

The code present in this notebook has been reused inside `main.py` file to create and populate the SQLite database - `stock_db.sqlite`.